# VLM 과제 Part 1: Small VLM 구조 이해

강의에서 제시한 Simple Mini VLM 코드를 기반으로  
**(1) [IMG_START]/[IMG_END] 특수 토큰 추가**,  
**(2) image token만 decoder에 입력**,  
**(3) learnable [IMG_SUM] 토큰 추가**  
세 가지 구조 변형을 실습합니다.


## 공통 Import

In [9]:
import torch
import torch.nn as nn


## 강의 베이스 코드 (수정 없음)

In [10]:
class MiniVisionEncoder(nn.Module):
    def __init__(self, image_size=224, image_channels=3, vision_dim=32, patch_size=16):
        super().__init__()
        self.patch_embed = nn.Conv2d(
            in_channels=image_channels,
            out_channels=vision_dim,
            kernel_size=patch_size,
            stride=patch_size,
        )
        num_patches = (image_size // patch_size) ** 2
        self.pos_embed = nn.Parameter(torch.randn(1, num_patches, vision_dim))
        layer = nn.TransformerEncoderLayer(
            d_model=vision_dim, nhead=4, batch_first=True, dropout=0.0, activation='gelu',
        )
        self.transformer = nn.TransformerEncoder(layer, num_layers=1)
        self.norm = nn.LayerNorm(vision_dim)

    def forward(self, pixel_values):
        x = self.patch_embed(pixel_values)
        image_features = x.flatten(2).transpose(1, 2)
        image_features = image_features + self.pos_embed
        image_features = self.transformer(image_features)
        image_features = self.norm(image_features)
        return image_features


class MiniProjector(nn.Module):
    def __init__(self, vision_dim=32, text_dim=64):
        super().__init__()
        self.proj = nn.Linear(vision_dim, text_dim)

    def forward(self, image_features):
        return self.proj(image_features)


---
## 과제 (1): [IMG_START] / [IMG_END] 특수 토큰 추가

**목표**

| 기존 | 변경 |
|------|------|
| `image_tokens + text_tokens` | `[IMG_START] + image_tokens + [IMG_END] + text_tokens` |

- vocabulary 크기를 2 늘려 `[IMG_START]`(id=1000), `[IMG_END]`(id=1001)를 추가합니다.
- 두 토큰은 각자 learnable embedding 벡터를 가집니다.


In [11]:
class VLM_with_IMG_boundary(nn.Module):
    """
    [IMG_START], [IMG_END] 두 개의 특수 토큰을 vocabulary에 추가하고
    decoder 입력 시퀀스를 구성할 때 image token 앞뒤에 삽입합니다.
    """
    def __init__(self, image_size=224, vocab_size=1000, vision_dim=32, text_dim=64, patch_size=16):
        super().__init__()
        # [IMG_START], [IMG_END]를 위해 vocab 2개 확장
        self.extended_vocab_size = vocab_size + 2
        self.img_start_id = vocab_size      # index 1000
        self.img_end_id   = vocab_size + 1  # index 1001

        self.vision_encoder = MiniVisionEncoder(image_size=image_size, vision_dim=vision_dim, patch_size=patch_size)
        self.projector      = MiniProjector(vision_dim=vision_dim, text_dim=text_dim)

        # 확장 vocab으로 token embedding 구성
        self.token_embed = nn.Embedding(self.extended_vocab_size, text_dim)
        layer = nn.TransformerEncoderLayer(
            d_model=text_dim, nhead=4, batch_first=True, dropout=0.0, activation='gelu'
        )
        self.decoder = nn.TransformerEncoder(layer, num_layers=1)
        self.lm_head = nn.Linear(text_dim, self.extended_vocab_size)

    def forward(self, pixel_values, input_ids):
        B = pixel_values.size(0)

        # 1. Vision Encoder + Projector
        image_features = self.vision_encoder(pixel_values)   # [B, N_img, D_vision]
        image_embeds   = self.projector(image_features)      # [B, N_img, D_text]

        # 2. [IMG_START] / [IMG_END] 임베딩
        img_start_ids = torch.full((B, 1), self.img_start_id, dtype=torch.long, device=pixel_values.device)
        img_end_ids   = torch.full((B, 1), self.img_end_id,   dtype=torch.long, device=pixel_values.device)
        img_start_emb = self.token_embed(img_start_ids)  # [B, 1, D_text]
        img_end_emb   = self.token_embed(img_end_ids)    # [B, 1, D_text]

        # 3. 텍스트 임베딩
        text_embeds = self.token_embed(input_ids)         # [B, L_text, D_text]

        # 4. 시퀀스 구성: [IMG_START] + image_tokens + [IMG_END] + text_tokens
        inputs_embeds = torch.cat(
            [img_start_emb, image_embeds, img_end_emb, text_embeds], dim=1
        )  # [B, 1 + N_img + 1 + L_text, D_text]

        # 5. Decoder
        seq_len = inputs_embeds.size(1)
        causal_mask = nn.Transformer.generate_square_subsequent_mask(seq_len).to(inputs_embeds.device)
        hidden_states = self.decoder(inputs_embeds, mask=causal_mask, is_causal=True)
        logits = self.lm_head(hidden_states)

        return logits, inputs_embeds


### 과제 (1) 실행 및 제출 결과 출력

In [12]:
B, H, W, L_TEXT = 2, 224, 224, 8
VOCAB, D_VISION, D_TEXT, PATCH = 1000, 32, 64, 16

torch.manual_seed(0)
pixel_values = torch.randn(B, 3, H, W)
input_ids    = torch.randint(0, VOCAB, (B, L_TEXT))

model_1 = VLM_with_IMG_boundary(
    image_size=H, vocab_size=VOCAB, vision_dim=D_VISION, text_dim=D_TEXT, patch_size=PATCH
)
model_1.eval()

with torch.no_grad():
    logits_1, inputs_embeds_1 = model_1(pixel_values, input_ids)

N_img            = (H // PATCH) ** 2          # 196
original_seq_len = N_img + L_TEXT             # 204
new_seq_len      = 1 + N_img + 1 + L_TEXT     # 206
seq_increase     = new_seq_len - original_seq_len

print("▶ decoder input shape :", tuple(inputs_embeds_1.shape))
print(f"  (B={B}, seq_len={new_seq_len}, D_text={D_TEXT})")

print("\n▶ sequence length 증가분")
print(f"  기존 seq_len : {original_seq_len}  (N_img={N_img} + L_text={L_TEXT})")
print(f"  변경 seq_len : {new_seq_len}  ([IMG_START]=1 + N_img={N_img} + [IMG_END]=1 + L_text={L_TEXT})")
print(f"  증가분       : +{seq_increase}")

first_img_pos = 1
last_img_pos  = N_img   # 1 + N_img - 1 = 196

print(f"\n▶ 첫 번째 image token 위치 : {first_img_pos}")
print(f"   마지막 image token 위치 : {last_img_pos}")
print(f"\n▶ 첫 번째 image token 값 (앞 5개 dim) :")
print(f"   inputs_embeds_1[0, {first_img_pos}, :5] = {inputs_embeds_1[0, first_img_pos, :5].tolist()}")
print(f"\n▶ 마지막 image token 값 (앞 5개 dim) :")
print(f"   inputs_embeds_1[0, {last_img_pos}, :5] = {inputs_embeds_1[0, last_img_pos, :5].tolist()}")

print("\n▶ token별 위치")
print(f"  {0}              : [IMG_START]")
print(f"  {first_img_pos} ~ {N_img}     : image tokens  ({N_img}개)")
print(f"  {N_img + 1}           : [IMG_END]")
print(f"  {N_img + 2} ~ {new_seq_len - 1}   : text tokens   ({L_TEXT}개)")


▶ decoder input shape : (2, 206, 64)
  (B=2, seq_len=206, D_text=64)

▶ sequence length 증가분
  기존 seq_len : 204  (N_img=196 + L_text=8)
  변경 seq_len : 206  ([IMG_START]=1 + N_img=196 + [IMG_END]=1 + L_text=8)
  증가분       : +2

▶ 첫 번째 image token 위치 : 1
   마지막 image token 위치 : 196

▶ 첫 번째 image token 값 (앞 5개 dim) :
   inputs_embeds_1[0, 1, :5] = [0.2564437985420227, 0.15369492769241333, -0.1810372769832611, -0.04668667912483215, -1.1985595226287842]

▶ 마지막 image token 값 (앞 5개 dim) :
   inputs_embeds_1[0, 196, :5] = [-0.8953401446342468, -0.43853580951690674, -0.8158922791481018, -1.0655525922775269, 0.20822374522686005]

▶ token별 위치
  0              : [IMG_START]
  1 ~ 196     : image tokens  (196개)
  197           : [IMG_END]
  198 ~ 205   : text tokens   (8개)


---
## 과제 (2): text 없이 image token만 decoder에 입력

**목표**

| 기존 | 변경 |
|------|------|
| `image_tokens + text_tokens` | `image_tokens only` |


In [13]:
class VLM_image_only(nn.Module):
    """text token 없이 image token만 decoder에 입력합니다."""
    def __init__(self, image_size=224, vocab_size=1000, vision_dim=32, text_dim=64, patch_size=16):
        super().__init__()
        self.vision_encoder = MiniVisionEncoder(image_size=image_size, vision_dim=vision_dim, patch_size=patch_size)
        self.projector      = MiniProjector(vision_dim=vision_dim, text_dim=text_dim)

        layer = nn.TransformerEncoderLayer(
            d_model=text_dim, nhead=4, batch_first=True, dropout=0.0, activation='gelu'
        )
        self.decoder = nn.TransformerEncoder(layer, num_layers=1)
        self.lm_head = nn.Linear(text_dim, vocab_size)

    def forward(self, pixel_values):
        # 1. Vision Encoder + Projector
        image_features = self.vision_encoder(pixel_values)  # [B, N_img, D_vision]
        image_embeds   = self.projector(image_features)     # [B, N_img, D_text]

        # 2. image token만 decoder에 입력 (text 없음)
        inputs_embeds = image_embeds  # [B, N_img, D_text]

        seq_len = inputs_embeds.size(1)
        causal_mask = nn.Transformer.generate_square_subsequent_mask(seq_len).to(inputs_embeds.device)
        hidden_states = self.decoder(inputs_embeds, mask=causal_mask, is_causal=True)
        logits = self.lm_head(hidden_states)

        return logits, inputs_embeds


### 과제 (2) 실행 및 제출 결과 출력

In [14]:
B, H, W, L_TEXT = 2, 224, 224, 8
VOCAB, D_VISION, D_TEXT, PATCH = 1000, 32, 64, 16

torch.manual_seed(0)
pixel_values = torch.randn(B, 3, H, W)

model_2 = VLM_image_only(
    image_size=H, vocab_size=VOCAB, vision_dim=D_VISION, text_dim=D_TEXT, patch_size=PATCH
)
model_2.eval()

with torch.no_grad():
    logits_2, inputs_embeds_2 = model_2(pixel_values)

N_img = (H // PATCH) ** 2  # 196

print("▶ decoder input shape :", tuple(inputs_embeds_2.shape))
print(f"  (B={B}, seq_len={N_img}, D_text={D_TEXT})")
print(f"\n▶ image token 개수 : {N_img}")
print(f"   text token 개수  : 0  (text 없음)")
print(f"   sequence length  : {N_img}  (image tokens only)")

print("\n▶ token별 위치")
print(f"  0 ~ {N_img - 1} : image tokens  ({N_img}개)")
print(f"  (text tokens 없음)")

print(f"\n▶ [비교] 기존 seq_len={N_img + L_TEXT} → 변경 seq_len={N_img}  (text {L_TEXT}개 제거)")


▶ decoder input shape : (2, 196, 64)
  (B=2, seq_len=196, D_text=64)

▶ image token 개수 : 196
   text token 개수  : 0  (text 없음)
   sequence length  : 196  (image tokens only)

▶ token별 위치
  0 ~ 195 : image tokens  (196개)
  (text tokens 없음)

▶ [비교] 기존 seq_len=204 → 변경 seq_len=196  (text 8개 제거)


---
## 과제 (3): learnable [IMG_SUM] 토큰 추가

**목표**

| 기존 | 변경 |
|------|------|
| `projected_image_tokens + text_tokens` | `[IMG_SUM] + projected_image_tokens + text_tokens` |

- `[IMG_SUM]`은 `nn.Parameter`로 선언된 **학습 가능한 벡터** 1개입니다.
- ViT의 `[CLS]` 토큰과 동일한 개념으로, self-attention을 통해 모든 image patch 정보를 집약합니다.
- 파라미터 증가분 = `D_text` = 64개


In [15]:
class VLM_with_IMG_SUM(nn.Module):
    """
    [IMG_SUM]: vision encoder 출력 앞에 삽입되는 learnable special token.
    - 이미지 전체 정보를 하나의 벡터로 요약 (ViT [CLS]와 유사)
    - nn.Parameter로 선언 → gradient로 학습됨
    """
    def __init__(self, image_size=224, vocab_size=1000, vision_dim=32, text_dim=64, patch_size=16):
        super().__init__()
        self.vision_encoder = MiniVisionEncoder(image_size=image_size, vision_dim=vision_dim, patch_size=patch_size)
        self.projector      = MiniProjector(vision_dim=vision_dim, text_dim=text_dim)

        # [IMG_SUM]: text_dim 크기의 learnable 벡터 1개
        self.img_sum_token = nn.Parameter(torch.randn(1, 1, text_dim))

        self.token_embed = nn.Embedding(vocab_size, text_dim)
        layer = nn.TransformerEncoderLayer(
            d_model=text_dim, nhead=4, batch_first=True, dropout=0.0, activation='gelu'
        )
        self.decoder = nn.TransformerEncoder(layer, num_layers=1)
        self.lm_head = nn.Linear(text_dim, vocab_size)

    def forward(self, pixel_values, input_ids):
        B = pixel_values.size(0)

        # 1. Vision Encoder + Projector
        image_features = self.vision_encoder(pixel_values)  # [B, N_img, D_vision]
        image_embeds   = self.projector(image_features)     # [B, N_img, D_text]

        # 2. [IMG_SUM] 토큰을 batch 크기에 맞게 확장
        img_sum = self.img_sum_token.expand(B, -1, -1)      # [B, 1, D_text]

        # 3. 텍스트 임베딩
        text_embeds = self.token_embed(input_ids)            # [B, L_text, D_text]

        # 4. 시퀀스 구성: [IMG_SUM] + projected_image_tokens + text_tokens
        inputs_embeds = torch.cat([img_sum, image_embeds, text_embeds], dim=1)
        # [B, 1 + N_img + L_text, D_text]

        # 5. Decoder
        seq_len = inputs_embeds.size(1)
        causal_mask = nn.Transformer.generate_square_subsequent_mask(seq_len).to(inputs_embeds.device)
        hidden_states = self.decoder(inputs_embeds, mask=causal_mask, is_causal=True)
        logits = self.lm_head(hidden_states)

        return logits, inputs_embeds


### 과제 (3) 실행 및 제출 결과 출력

In [ ]:
B, H, W, L_TEXT = 2, 224, 224, 8
VOCAB, D_VISION, D_TEXT, PATCH = 1000, 32, 64, 16

torch.manual_seed(0)
pixel_values = torch.randn(B, 3, H, W)
input_ids    = torch.randint(0, VOCAB, (B, L_TEXT))

# 파라미터 비교용 베이스 모델
class _BaseVLM(nn.Module):
    def __init__(self):
        super().__init__()
        self.vision_encoder = MiniVisionEncoder(image_size=H, vision_dim=D_VISION, patch_size=PATCH)
        self.projector      = MiniProjector(vision_dim=D_VISION, text_dim=D_TEXT)
        self.token_embed    = nn.Embedding(VOCAB, D_TEXT)
        layer = nn.TransformerEncoderLayer(d_model=D_TEXT, nhead=4, batch_first=True, dropout=0.0, activation='gelu')
        self.decoder = nn.TransformerEncoder(layer, num_layers=1)
        self.lm_head = nn.Linear(D_TEXT, VOCAB)

base_model = _BaseVLM()

model_3 = VLM_with_IMG_SUM(
    image_size=H, vocab_size=VOCAB, vision_dim=D_VISION, text_dim=D_TEXT, patch_size=PATCH
)
model_3.eval()

with torch.no_grad():
    logits_3, inputs_embeds_3 = model_3(pixel_values, input_ids)

N_img       = (H // PATCH) ** 2   # 196
new_seq_len = 1 + N_img + L_TEXT  # 205

def count_params(model):
    total     = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, trainable

base_total, base_trainable = count_params(base_model)
new_total,  new_trainable  = count_params(model_3)

print("▶ decoder input shape :", tuple(inputs_embeds_3.shape))
print(f"  (B={B}, seq_len={new_seq_len}, D_text={D_TEXT})")

print("\n▶ 첫 번째 token 값 (batch 0, 앞 5개 dim) :")
print(f"  inputs_embeds_3[0, 0, :5] = {inputs_embeds_3[0, 0, :5].tolist()}")
print(f"  → 이 값이 learnable [IMG_SUM] 토큰 임베딩입니다.")

print("\n▶ parameter 수 비교")
print(f"  베이스 모델  총 파라미터 : {base_total:,}")
print(f"  IMG_SUM 모델 총 파라미터 : {new_total:,}")
print(f"  증가한 파라미터          : +{new_total - base_total:,}  ([IMG_SUM] = 1 × D_text({D_TEXT}) = {D_TEXT}개)")

print("\n▶ token별 위치")
print(f"  0              : [IMG_SUM]   ← learnable special token")
print(f"  1 ~ {N_img}     : image tokens  ({N_img}개)")
print(f"  {N_img + 1} ~ {new_seq_len - 1}   : text tokens   ({L_TEXT}개)")

print("\n▶ [IMG_SUM] 역할")
print("  - Attention을 통해 모든 image patch 정보를 집약하는 요약 토큰")
print("  - ViT의 [CLS] 토큰과 개념적으로 동일한 역할")
print("  - 학습 과정에서 이미지 전체의 의미를 압축하도록 gradient로 업데이트됨")
print("  - text decoder가 이미지를 이해할 때 개별 patch 대신 이 토큰을 참조할 수 있음")


▶ decoder input shape : (2, 205, 64)
  (B=2, seq_len=205, D_text=64)

▶ 첫 번째 token 값 (batch 0, 앞 5개 dim) :
  inputs_embeds_3[0, 0, :5] = [-0.4657396674156189, 0.5400055050849915, -0.7680185437202454, 1.4902923107147217, 0.8603426814079285]
  → 이 값이 learnable [IMG_SUM] 토큰 임베딩입니다.

▶ parameter 수 비교
  베이스 모델  총 파라미터 : 580,712
  IMG_SUM 모델 총 파라미터 : 580,776
  증가한 파라미터          : +64  ([IMG_SUM] = 1 × D_text(64) = 64개)

▶ token별 위치
  0              : [IMG_SUM]   ← learnable special token
  1 ~ 196     : image tokens  (196개)
  197 ~ 204   : text tokens   (8개)

▶ [IMG_SUM] 역할
  - Attention을 통해 모든 image patch 정보를 집약하는 요약 토큰
  - ViT의 [CLS] 토큰과 개념적으로 동일한 역할
  - 학습 과정에서 이미지 전체의 의미를 압축하도록 gradient로 업데이트됨
  - text decoder가 이미지를 이해할 때 개별 patch 대신 이 토큰을 참조할 수 있음


: 